# Module Nine Project

For the module nine project you are required to produce coding samples in accordance with the requirements in the project guidelines and rubric. This Notebook has been pre-created as a shell for your project submission.

Once your code is complete, you will download this notebook as an HTML file by clicking **File**, then **Download As**, then **HTML (.html)** from the Jupyter menu bar. Validate that your code and outputs are rendered in the html file before submission.

<div class="alert alert-block alert-danger">
    <span style="color:black"><b><i>Missing Blocks:</i></b> Ensure that you have clicked <b><mark>Run</mark></b> for all blocks before downloading as an HTML file or the block may be missing from your submission.</span>
</div>

### Data Source Information
Below are some example reputable data sources that you can use to collect your minimum of two data sets from:

- https://datasetsearch.research.google.com/
- https://www.kaggle.com/datasets
- https://data.gov/
- https://www.census.gov/data.html
- https://www.bls.gov/data/
- https://databank.worldbank.org/databases
- https://www.ncei.noaa.gov/access/search/index
- https://www.epa.gov/data
- https://www.pewresearch.org/tools-and-datasets/
- https://www.who.int/data/
- https://dataverse.harvard.edu/
- https://search.cdc.gov/search/?query=dataset&dpage=1
- https://data.fivethirtyeight.com/
- https://www.gdeltproject.org/data.html
- https://datahub.io/collections


***Note: this list is not all inclusive, if you have a reputable data set that you have collected, or would like to use, please ensure to upload the data. All data sets will need to be documented in your submission as to the source through the creation of a citation.***

<div class="alert alert-block alert-success">
    <span style="color:black"><b><i>To Begin:</i></b> replace each <b>data_file_#</b> entry sample data path with the path to your selected data sets.</span>
</div>

You are welcome to use more than two data sets, create an entry for each as applicable for each data set. Ensure that you create comments that provide citations for the data sets you have selected for use. When working through your submission, keep in mind that you will need to have choosen two separate groups of stakeholders to create your final visualizations.

## Installing Required Packages
If you have additional packages that you would like to use or install, add them to the **requirements.txt** file then run the next two blocks.

In [28]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [29]:
%pip install jupyter nbconvert nbformat

Note: you may need to restart the kernel to use updated packages.


In [30]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [31]:
%pip install plotly numpy

Note: you may need to restart the kernel to use updated packages.


In [32]:
%pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [33]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


## Import the Required Packages

In [34]:
import pandas as pd
import matplotlib as plt
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import statsmodels.api as sm
import plotly.io as pio
from IPython.display import IFrame, display # for image rendering in html output

pd.set_option('display.max_columns', None)

## Loading the Data
Create entries to load your data for your submission.

In [35]:
# Placeholder for data citation information
data_file_1 = "NBA_Data/Draft Pick History.csv"
data_file_2 = "NBA_Data/Team Totals.csv"
data_file_3 = "NBA_Data/standings.csv"
data_file_4 = "NBA_Data/Advanced.csv"

draft = pd.read_csv(data_file_1)
team_lookup = pd.read_csv(data_file_2)
standings = pd.read_csv(data_file_3)
player_stats = pd.read_csv(data_file_4)


## Examining the Data
Create entries to conduct exploratory data analysis to identify key data points to include relevant data from your two data sets while evaluating the quality of your choosen data.

In [36]:
eda_frames = {
	"draft": draft,
	"team_lookup": team_lookup,
	"standings": standings,
	"player_stats": player_stats
}

def first_existing_col(df, candidates):
	return next((col for col in candidates if col in df.columns), None)

def summarize_df(name, df):
	season_col = first_existing_col(df, ["season", "draft_year"])
	team_col = first_existing_col(df, ["team", "draft_team", "team_abbrev", "draft_team_abbrev"])
	player_col = first_existing_col(df, ["player_id", "player", "rookie_player"])
	
	return {
		"dataset": name,
		"rows": len(df),
		"columns": df.shape[1],
		"duplicate_rows": int(df.duplicated().sum()),
		"missing_cells": int(df.isna().sum().sum()),
		"missing_pct": round((df.isna().sum().sum() / df.size) * 100, 2),
		"start_year": int(df[season_col].min()) if season_col else np.nan,
		"end_year": int(df[season_col].max()) if season_col else np.nan,
		"unique_teams": int(df[team_col].nunique()) if team_col else np.nan,
		"unique_players": int(df[player_col].nunique()) if player_col else np.nan,
	}

eda_overview = pd.DataFrame([summarize_df(name, df) for name, df in eda_frames.items()])
print("High-level dataset overview")
display(eda_overview)

for name, df in eda_frames.items():
	print(f"\n{name.upper()} — sample records")
	display(df.head(3))
	
	missing_summary = (
		pd.DataFrame({
			"missing_count": df.isna().sum(),
			"missing_pct": (df.isna().mean() * 100).round(2)
		})
		.query("missing_count > 0")
		.sort_values("missing_pct", ascending=False)
		.head(10)
	)
	
	if not missing_summary.empty:
		print(f"Top missing-value columns in {name}")
		display(missing_summary)

draft_summary = (
	draft.groupby("season", as_index=False)
	.agg(
		total_picks=("player_id", "count"),
		first_round_picks=("round", lambda s: (s == 1).sum()),
		second_round_picks=("round", lambda s: (s == 2).sum()),
		teams_with_picks=("tm", "nunique"),
		avg_overall_pick=("overall_pick", "mean")
	)
	.round(2)
	.sort_values("season", ascending=False)
)

print("\nDraft summary by year")
display(draft_summary.head(10))

team_lookup_summary = (
	team_lookup.groupby("season", as_index=False)
	.agg(
		teams=("team", "nunique"),
		abbreviations=("abbreviation", "nunique"),
		playoff_teams=("playoffs", "sum")
	)
	.sort_values("season", ascending=False)
)

print("\nTeam lookup summary by season")
display(team_lookup_summary.head(10))

standings_summary = (
	standings.groupby(["Conference"], as_index=False)
	.agg(
		teams=("TeamID", "nunique"),
		avg_wins=("WINS", "mean"),
		avg_losses=("LOSSES", "mean"),
		avg_winpct=("WinPCT", "mean")
	)
	.round(3)
)

print("\nStandings summary by conference and playoff status")
display(standings_summary)

player_stats_summary = (
	player_stats.groupby("player", observed=False, as_index=False)
	.agg(
		unique_players=("player_id", "nunique"),
		career_avg_vorp=("vorp", "mean"),
		median_vorp=("vorp", "median"),
		seasons_played=("season", "nunique"),
		rookie_age=("age", "min")
	)
	.round(2)
)

print("\nPlayer stats summary by draft pick tier")
display(player_stats_summary)

print("\nKey numeric summary for player_stats")
display(
	player_stats[
		["player", "g", "mp", "bpm", "vorp", "season", "age"]
	].describe().round(2)
)

High-level dataset overview


,dataset,rows,columns,duplicate_rows,missing_cells,missing_pct,start_year,end_year,unique_teams,unique_players
0,draft,8383,8,0,885,1.32,1947,2025,NaN,8225.0
1,team_lookup,1907,28,0,3369,6.31,1946,2025,97.0,NaN
2,standings,1712,10,0,228,1.33,1947,2025,NaN,NaN
3,player_stats,25680,30,0,1779,0.23,1978,2025,45.0,3862.0



DRAFT — sample records


,season,lg,overall_pick,round,tm,player,player_id,college
0,2025,NBA,1.0,1.0,DAL,Cooper Flagg,flaggco01,Duke
1,2025,NBA,2.0,1.0,SAS,Dylan Harper,harpedy01,Rutgers University
2,2025,NBA,3.0,1.0,PHI,VJ Edgecombe,edgecvj01,Baylor


Top missing-value columns in draft


...

,Conference,teams,avg_wins,avg_losses,avg_winpct
0,East,21,39.328,41.135,0.489
1,West,22,41.178,39.376,0.511



Player stats summary by draft pick tier


,player,unique_players,career_avg_vorp,median_vorp,seasons_played,rookie_age
0,A.C. Green,1,0.95,0.90,16,22.0
1,A.J. Bramlett,1,-0.20,-0.20,1,23.0
2,A.J. English,1,-0.75,-0.75,2,23.0
3,A.J. Green,1,-0.20,-0.05,4,23.0
4,A.J. Guyton,1,-0.07,0.00,3,22.0
...,...,...,...,...,...,...
3833,Šarūnas Marčiulionis,1,0.67,0.50,7,25.0
3834,Žan Tabak,1,-0.46,-0.35,6,24.0
3835,Žarko Paspalj,1,-0.40,-0.40,1,23.0
3836,Žarko Čabarkapa,1,-0.10,0.00,3,22.0



Key numeric summary for player_stats


,g,mp,bpm,vorp,season,age
count,25680.00,25680.00,25675.00,25680.00,25680.00,25680.00
mean,47.36,1099.21,-1.90,0.53,2004.43,26.62
std,26.60,895.97,5.04,1.25,13.56,4.04
min,1.00,0.00,-92.10,-2.60,1978.00,18.00
25%,23.00,281.00,-3.70,-0.10,1993.00,24.00
50%,51.00,912.00,-1.50,0.00,2006.00,26.00
75%,73.00,1790.00,0.40,0.80,2016.00,29.00
max,85.00,3533.00,242.20,12.50,2025.00,44.00


## Data Preprocessing Techniques
Create entries to clean and blend your data sets as applicable consolidating the two or more data sets into a single collection for inclusion into your visualizations.

In [37]:
# get post-merger data
draft = draft[draft['season'] >= 1978].copy()
team_lookup = team_lookup[team_lookup['season'] >= 1978].copy()
standings = standings[standings['season'] >= 1978].copy()
player_stats = player_stats[player_stats['season'] >= 1978].copy()

team_lookup = team_lookup[team_lookup['team'] != 'League Average'].copy() # drop league average rows from team lookup since they are not actual teams and will cause issues with mapping
# clean up dataframe
standings.loc[standings['TeamCity']=='LA', 'TeamCity'] = 'Los Angeles'
standings['team'] = standings['TeamCity'] + ' ' + standings['TeamName']
standings['TeamID'] = standings['TeamID'].astype(str)
standings.drop(columns=['TeamCity','TeamName','Division'], inplace=True)
standings.rename(columns={'PlayoffRank':'ConferenceRank'}, inplace=True)

# change hornets to pelicans prior to 2002 for the sake of accurate draft analysis
# since the pelicans technically remained the hornets franchise for the sake of draft history and maintained players
mask = ((standings['team'] == 'Charlotte Hornets') & (standings['season'] <= 2002 ))
standings.loc[mask, 'TeamID'] = '1610612740'

# get first & second round draft picks
draft = draft[draft['round'] <= 2]

# split dataset between full season stats and partial season stats for players that were traded mid-season
player_stats['team'] = player_stats.sort_values('season').groupby('player')['team'].transform(lambda x: np.where(x.str.contains(r'^\dTM$', na=False), x.shift(-1), x))
idx = player_stats.groupby(['player_id', 'season'])['g'].idxmax()
player_stats_partial_seasons = player_stats.drop(idx)
player_stats = player_stats.loc[idx].copy()

# drop unnecessary columns and rename for consistency
draft.drop(columns=['college','lg'], inplace=True)
draft.rename(columns={'tm':'team_abbrev','player':'rookie_player'}, inplace=True)
player_stats.rename(columns={'team':'team_abbrev'}, inplace=True)
team_lookup.rename(columns={'abbreviation':'team_abbrev'}, inplace=True)

# create lookup table
team_lookup = (team_lookup[['season', 'team', 'team_abbrev','playoffs']].merge(standings[['season', 'TeamID','team']].drop_duplicates(),on=['season', 'team'],how='left') [['season', 'TeamID', 'team', 'team_abbrev','playoffs']].drop_duplicates())

# modern mapping keyed by TeamID
team_map = (team_lookup[team_lookup['season'] == 2024].drop_duplicates('TeamID').set_index('TeamID')[['team', 'team_abbrev']])

# map text columns from TeamID
standings['team'] = standings['TeamID'].map(team_map['team'])
standings['team_abbrev'] = standings['TeamID'].map(team_map['team_abbrev'])

standings = standings.merge(team_lookup[['season', 'TeamID', 'playoffs']],on=['season', 'TeamID'],how='left')
draft = (draft.merge(team_lookup[['season', 'team_abbrev', 'TeamID', 'team']],on=['season', 'team_abbrev'],how='left').rename(columns={'TeamID': 'draft_team_ID', 'team': 'draft_team','season': 'draft_year','team_abbrev': 'draft_team_abbrev'}))
player_stats = player_stats.merge(team_lookup[['season', 'team_abbrev', 'TeamID', 'team']],on=['season', 'team_abbrev'],how='left')

player_stats['team'] = player_stats['TeamID'].map(team_map['team'])
player_stats['team_abbrev'] = player_stats['TeamID'].map(team_map['team_abbrev'])
draft['draft_team'] = draft['draft_team_ID'].map(team_map['team'])
draft['draft_team_abbrev'] = draft['draft_team_ID'].map(team_map['team_abbrev'])

# only bring in the draft columns you actually need
result = (
    pd.concat([player_stats[['player_id','season']], draft[['player_id','draft_year']].rename(columns={'draft_year':'season'})], ignore_index=True)
    .drop_duplicates(subset=["season", "player_id"], keep="first")
)

result = result.merge(player_stats, on=['player_id','season'], how='left')
result = result.merge(draft[['player_id', 'draft_year', 'draft_team_ID', 'draft_team','draft_team_abbrev', 'overall_pick','round']], left_on=['player_id'], right_on=['player_id'], how='left')
player_stats = result.copy()
player_stats = player_stats.merge(draft[['player_id', 'draft_team_ID', 'draft_team', 'overall_pick','draft_team_abbrev','round']], on=['player_id','draft_team_ID','draft_team_abbrev','draft_team', 'overall_pick','round'], how='left')
player_stats['player'] = player_stats.groupby('player_id')['player'].transform(lambda x: x.ffill().bfill() if x.isna().any() else x)
player_stats['TeamID'] = player_stats['TeamID'].fillna(player_stats['draft_team_ID'])
player_stats['team'] = player_stats['team'].fillna(player_stats['draft_team'])
player_stats['team_abbrev'] = player_stats['team_abbrev'].fillna(player_stats['draft_team_abbrev'])

# calulate career stats
player_stats['vorp'] = player_stats['vorp'].fillna(0)
player_stats['career_total_vorp']=player_stats.groupby('player_id')['vorp'].transform('sum').round(3)
player_stats['career_avg_vorp']=player_stats.groupby('player_id')['vorp'].transform('mean').round(3)
player_stats['career_median_vorp']=player_stats.groupby('player_id')['vorp'].transform('median').round(3)
player_stats['career_g']=player_stats.groupby('player_id')['g'].transform('sum')
player_stats['seasons_played'] = player_stats.groupby('player_id')['season'].transform('count')
player_stats['rookie_age'] = player_stats.groupby('player_id')['age'].transform('min').round(3)

#calculate players average vorp for first five season
first_five = player_stats.sort_values(['player_id', 'season']).groupby('player_id').head(5).groupby('player_id', as_index=False)['vorp'].sum().rename(columns={'vorp': "first_five_vorp"})[['player_id','first_five_vorp']]
player_stats = player_stats.merge(first_five, on='player_id', how="left")
player_stats['first_five_vorp'] = player_stats['first_five_vorp'].round(3)

player_stats = player_stats[['season','player','player_id','team_abbrev','TeamID','team','draft_year','draft_team_ID','draft_team_abbrev','draft_team','overall_pick','round','pos','g','gs','mp','bpm','vorp','first_five_vorp','career_total_vorp','career_avg_vorp','career_median_vorp','career_g','seasons_played','age','rookie_age']]
# create pick tier column based on overall pick
tier_order = ["1-3", "4-7", "8-14", "15-20", "21-30"]
player_stats["pick_tier"] = pd.cut(player_stats["overall_pick"],bins=[0, 3, 7, 14, 20, 30],labels=tier_order)
player_stats["pick_tier"] = pd.Categorical(player_stats["pick_tier"],categories=tier_order,ordered=True)

# combine player and team data into single dataframe for analysis
player_stats = player_stats[player_stats['season'].between(1978, 2024)].copy()
standings = standings[standings['season'].between(1978, 2024)].copy()
data = standings.merge(player_stats, on=['season','TeamID','team','team_abbrev'], how='outer')

# get one record per player and calculate draft class strength metrics
# does not include standings data for teams that did not have a first round pick in a given year
draft_class_data = data[data['season']==data['draft_year']].copy()
draft_class_data = draft_class_data[draft_class_data['player_id'].isin(draft['player_id'])]
draft_class_data.rename(columns={'player':'rookie_player'}, inplace=True)
draft_class_data = draft_class_data.fillna({'rookie_player': 'No Rookie'})
draft_class_data = draft_class_data.fillna({'draft_team': draft_class_data['team'],'draft_team_ID': draft_class_data['TeamID'],'draft_team_abbrev': draft_class_data['team_abbrev']})
draft_class_data = draft_class_data[draft_class_data['draft_year'] <= 2021] # only include players drafted through 2021 to give all players at least 4 full seasons
draft_class_data = draft_class_data[draft_class_data['round'] == 1] # only include first round picks for draft class analysis

draft_class_data['all_players_career_avg_vorp'] = draft_class_data['career_avg_vorp'].mean().round(3)
draft_class_data["all_players_std_career_avg_vorp"] = draft_class_data["career_avg_vorp"].std()
draft_class_data["all_players_vorp_zscore"] = ((draft_class_data["career_avg_vorp"] - draft_class_data["all_players_career_avg_vorp"]) /draft_class_data["all_players_std_career_avg_vorp"]).replace([np.inf, -np.inf], np.nan).round(2)

draft_class_data['draft_class_rookie_age'] = draft_class_data.groupby('draft_year')['rookie_age'].transform('mean').round(3)
draft_class_data['draft_class_mean_ff_vorp'] = draft_class_data.groupby("season")['first_five_vorp'].transform('mean').round(2)
draft_class_data['draft_class_total_career_total_vorp']= draft_class_data.groupby("season")['career_total_vorp'].transform('sum').round(2)
draft_class_data['draft_class_mean_career_total_vorp'] = draft_class_data.groupby("season")['career_total_vorp'].transform('mean').round(2)
draft_class_data['draft_class_median_career_total_vorp'] = draft_class_data.groupby("season")['career_total_vorp'].transform('median').round(2)
draft_class_data['draft_class_mean_career_avg_vorp'] = draft_class_data.groupby("season")['career_avg_vorp'].transform('mean').round(2)
draft_class_data['draft_class_size'] = draft_class_data.groupby("season")['overall_pick'].transform('nunique')
draft_class_data["draft_class_std_ff_vorp"] = draft_class_data.groupby("season")["first_five_vorp"].transform("std")
draft_class_data["draft_class_std_career_avg_vorp"] = draft_class_data.groupby("season")["career_avg_vorp"].transform("std")
draft_class_data["draft_class_career_avg_vorp_above_all_players_avg"] = draft_class_data["draft_class_mean_career_avg_vorp"] - draft_class_data["all_players_career_avg_vorp"]
draft_class_data["draft_class_ff_vorp_zscore"] = ((draft_class_data["first_five_vorp"] - draft_class_data["draft_class_mean_ff_vorp"]) /draft_class_data["draft_class_std_ff_vorp"]).replace([np.inf, -np.inf], np.nan).round(2)
draft_class_data["draft_class_career_avg_vorp_zscore"] = ((draft_class_data["career_avg_vorp"] - draft_class_data["draft_class_mean_career_avg_vorp"]) /draft_class_data["draft_class_std_career_avg_vorp"]).replace([np.inf, -np.inf], np.nan).round(2)

draft_class_data["ff_vorp_above_class_avg"] = draft_class_data["first_five_vorp"] - draft_class_data["draft_class_mean_ff_vorp"]
draft_class_data["ff_vorp_vs_class_ratio"] = ((draft_class_data["first_five_vorp"] / draft_class_data["draft_class_mean_ff_vorp"]).replace([np.inf, -np.inf], np.nan).round(2))

draft_class_data['draft_pick_avg_zscore_vs_class'] = draft_class_data.groupby('overall_pick')['draft_class_career_avg_vorp_zscore'].transform('mean')
draft_class_data['draft_class_avg_zscore_vs_all_players'] = draft_class_data.groupby('season')['draft_class_career_avg_vorp_zscore'].transform('mean')

draft_class_data['player_zscore_diff_from_pick_avg'] = draft_class_data['draft_class_career_avg_vorp_zscore'] - draft_class_data['draft_pick_avg_zscore_vs_class']

pick_value_data = (draft_class_data.groupby(["overall_pick","pick_tier"], as_index=False).agg(
        pick_mean_ff_vorp=("first_five_vorp", "mean"),
        pick_median_ff_vorp=("first_five_vorp", "median"),
        pick_mean_ff_vorp_above_class_avg=("ff_vorp_above_class_avg", "mean"),
        pick_mean_ff_vorp_vs_class_ratio=("ff_vorp_vs_class_ratio", "mean"),
        pick_mean_career_avg_vorp=("career_avg_vorp", "mean"),
        pick_mean_career_total_vorp=("career_total_vorp", "mean"),
        pick_median_career_total_vorp=("career_total_vorp", "median"),
    ).sort_values("overall_pick")).round(2)

team_vorp_all_players = data.groupby(["season", "team"], as_index=False).agg(team_vorp=("vorp", "sum"), n_players=("player_id", "nunique"))
team_winpct_all_players = data[["season", "team", "WinPCT", "Conference", "WINS", "LOSSES","playoffs"]].drop_duplicates()

data.head(10)

,TeamID,Conference,WINS,LOSSES,WinPCT,ConferenceRank,season,team,team_abbrev,playoffs,player,player_id,draft_year,draft_team_ID,draft_team_abbrev,draft_team,overall_pick,round,pos,g,gs,mp,bpm,vorp,first_five_vorp,career_total_vorp,career_avg_vorp,career_median_vorp,career_g,seasons_played,age,rookie_age,pick_tier
0,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Charlie Criss,crissch01,NaN,NaN,NaN,NaN,NaN,NaN,PG,54.0,1.0,879.0,-3.2,-0.3,1.0,1.1,0.157,0.20,341.0,7,30.0,30.0,NaN
1,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,John Drew,drewjo01,NaN,NaN,NaN,NaN,NaN,NaN,SF,79.0,79.0,2410.0,2.1,2.5,6.1,6.8,0.971,0.90,440.0,7,24.0,24.0,NaN
2,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Jack Givens,givenja01,1978.0,1610612737,ATL,Atlanta Hawks,16.0,1.0,SF,74.0,3.0,1347.0,-0.7,0.5,0.6,0.6,0.300,0.30,156.0,2,22.0,22.0,15-20
3,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Steve Hawes,hawesst01,NaN,NaN,NaN,NaN,NaN,NaN,PF,81.0,54.0,2205.0,1.0,1.7,5.8,5.8,0.967,1.35,442.0,6,28.0,28.0,NaN
4,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Keith Herron,herroke01,1978.0,1610612757,POR,Portland Trail Blazers,24.0,2.0,SG,14.0,0.0,81.0,-4.8,-0.1,0.0,0.0,0.000,-0.10,124.0,3,22.0,22.0,21-30
5,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Armond Hill,hillar01,NaN,NaN,NaN,NaN,NaN,NaN,PG,82.0,82.0,2527.0,-1.4,0.4,-0.6,-0.8,-0.133,-0.10,305.0,6,25.0,25.0,NaN
6,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Eddie Johnson,johnsed02,NaN,NaN,NaN,NaN,NaN,NaN,SG,78.0,77.0,2413.0,1.2,2.0,10.3,11.6,1.289,0.80,596.0,9,23.0,23.0,NaN
7,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Butch Lee,leebu01,1978.0,1610612737,ATL,Atlanta Hawks,10.0,1.0,PG,82.0,4.0,1779.0,-1.0,0.4,0.3,0.3,0.150,0.15,96.0,2,22.0,22.0,8-14
8,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Tom McMillen,mcmilto01,NaN,NaN,NaN,NaN,NaN,NaN,C,82.0,0.0,1392.0,-2.2,-0.1,0.9,-0.1,-0.013,-0.10,535.0,8,26.0,26.0,NaN
9,1610612737,East,46.0,36.0,0.561,5.0,1978,Atlanta Hawks,ATL,True,Tree Rollins,rollitr01,NaN,NaN,NaN,NaN,NaN,NaN,C,81.0,45.0,1900.0,2.4,2.1,11.6,17.4,1.024,1.00,1076.0,17,23.0,23.0,NaN


## Data Visualizations
Create entries to visualize your data using appropriate visualization types to reveal key trends within the data collection. Integrate multiple data attributes into visualizations using appropriate techniques to derive key insights from your visualizations. In your data visualizations, you will use multi-level data modeling practices to refine your visualizations. Your visualizations should include a high-level visualization that provides a broad overview of the data for one of your two stakeholders, as well as a detailed, insight-focused visualization that highlights the key findings for the other of your two stakeholders.

In [38]:
team_draft_value = (
    draft_class_data.sort_values(["season", "team", "overall_pick"])
    .groupby(["season", "team"], as_index=False)
    .agg(
        n_first_round_picks=("rookie_player", "count"),
        mean_first_five_vorp=("first_five_vorp", "mean"),
        best_pick=("overall_pick", "min"),
        best_pick_tier=("pick_tier", "min"),
        total_first_five_vorp=("first_five_vorp", "sum"),
        total_career_vorp=("career_total_vorp", "sum"),
        best_picks = ("overall_pick",list),
        picks_and_players=("overall_pick",lambda s: ", ".join(f"#{pick} {player}"for pick, player in zip(s.astype(int),draft_class_data.loc[s.index, "rookie_player"]))),
    )
)
team_draft_value["best_pick_tier"] = pd.cut(
    team_draft_value["best_pick"],
    bins=[0, 3, 7, 14, 20, 30],
    labels=["1-3", "4-7", "8-14", "15-20","21-30"]
)

team_success_window = standings[["season", "team", "WinPCT", "playoffs"]].sort_values(["team", "season"]).copy()
team_success_window["playoffs_int"] = team_success_window["playoffs"].astype(int)

for i in range(5):
    team_success_window[f"winpct_plus_{i}"] = team_success_window.groupby("team")["WinPCT"].shift(-i)
    team_success_window[f"playoffs_plus_{i}"] = team_success_window.groupby("team")["playoffs_int"].shift(-i)

team_success_window["next_5yr_winpct"] = team_success_window[[f"winpct_plus_{i}" for i in range(5)]].mean(axis=1)
team_success_window["prev_1yr_winpct"] = team_success_window.groupby("team")["WinPCT"].shift(1)
team_success_window["winpct_change_after_draft"] = (team_success_window["next_5yr_winpct"] - team_success_window["prev_1yr_winpct"])
team_success_window["next_5yr_playoff_apps"] = team_success_window[[f"playoffs_plus_{i}" for i in range(5)]].sum(axis=1)

team_draft_impact = team_success_window.merge(team_draft_value, on=["season", "team"], how="left")
team_draft_impact = team_draft_impact[team_draft_impact["n_first_round_picks"].notna()]

tier_order = ["1-3", "4-7", "8-14", "15-20","21-30"]
tier_num_map = {"1-3": 1, "4-7": 2, "8-14": 3, "15-20": 4, "21-30": 5}

corr_data = team_draft_impact.copy()
corr_data["best_pick_tier_num"] = corr_data["best_pick_tier"].map(tier_num_map)

r = corr_data["best_pick_tier_num"].corr(corr_data["winpct_change_after_draft"])

fig_team_draft_impact = px.strip(
    corr_data,
    x="best_pick_tier",
    y="winpct_change_after_draft",
    color="best_pick_tier",
    category_orders={"best_pick_tier": tier_order},
    color_discrete_sequence=px.colors.qualitative.Safe,
    hover_name="team",
    custom_data=[
        "season",
        "picks_and_players",
        "prev_1yr_winpct",
        "next_5yr_winpct",
        "total_career_vorp",
        "total_first_five_vorp",
        "best_pick"
    ],
    title=f"Team Improvement by Best Pick Tier (r = {r:.2f})",
    labels={
        "best_pick_tier": "Best Pick Tier",
        "winpct_change_after_draft": "Change in Team Win% (Next 5 Years vs Previous 1 Year)"
    }
)

fig_team_draft_impact.update_traces(
    jitter=0.28,
    marker=dict(size=10, opacity=0.75),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Draft Year: %{customdata[0]}<br>"
        "Best Pick: %{customdata[6]}<br>"
        "Best Pick Tier: %{x}<br>"
        "Picks: %{customdata[1]}<br>"
        "Total First 5-Year VORP: %{customdata[5]:.2f}<br>"
        "Total Career VORP: %{customdata[4]:.2f}<br>"
        "Prev Year Win%: %{customdata[2]:.1%}<br>"
        "Next 5-Year Win%: %{customdata[3]:.1%}<br>"
        "Win% Change: %{y:.1%}<br>"
        "<extra></extra>"
    ),
    showlegend=False
)

# Group means
tier_means = (
    corr_data.groupby("best_pick_tier", observed=False)["winpct_change_after_draft"]
    .mean()
    .reindex(tier_order)
    .reset_index()
)

fig_team_draft_impact.add_trace(
    go.Scatter(
        x=tier_means["best_pick_tier"],
        y=tier_means["winpct_change_after_draft"],
        mode="markers+lines",
        name="Tier Mean",
        marker=dict(color="black", size=12, symbol="diamond"),
        line=dict(color="black", width=2, dash="dash"),
        hovertemplate="Tier: %{x}<br>Mean Win% Change: %{y:.1%}<extra></extra>",
        showlegend=False
    )
)

fig_team_draft_impact.add_hline(y=0, line_dash="dash", line_color="gray")

fig_team_draft_impact.update_layout(
    template="plotly_white",
    height=650,
    width=1300
)
fig_team_draft_impact.update_yaxes(tickformat=".0%")

fig_team_draft_impact.show()
#fig_team_draft_impact.write_html("team_draft_impact_win_percentage.html")
#display(IFrame("team_draft_impact_win_percentage.html", width=1300, height=650))


In [39]:
team_vorp_scatter_data = (
    team_vorp_all_players
    .merge(team_winpct_all_players, on=["season", "team"], how="inner")
    .dropna(subset=["team_vorp", "WinPCT"])
    .sort_values(["season", "team"])
)

r = team_vorp_scatter_data["team_vorp"].corr(team_vorp_scatter_data["WinPCT"])

fig_player_team_winpct = px.scatter(
    team_vorp_scatter_data,
    x="team_vorp",
    y="WinPCT",
    color="playoffs",
    size="n_players",
    size_max=16,
    opacity=0.75,
    trendline="ols",
    hover_name="team",
    custom_data=["season", "team", "WINS", "LOSSES", "n_players"],
    labels={
        "team_vorp": "Team Total VORP (All Players)",
        "WinPCT": "Team Winning Percentage",
        "Conference": "Conference",
        "n_players": "Players Counted"
    },
    title=f"Team Total VORP vs Team Win% (All Players, r = {r:.2f})"
)

fig_player_team_winpct.update_traces(
    selector=dict(mode="markers"),
    marker=dict(line=dict(width=0.5, color="white")),
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Season: %{customdata[0]}<br>"
        "Team: %{customdata[1]}<br>"
        "Team Total VORP: %{x:.2f}<br>"
        "Team Win%: %{y:.3f}<br>"
        "Record: %{customdata[2]}-%{customdata[3]}<br>"
        "Players Counted: %{customdata[4]}<br>"
        "<extra></extra>"
    )
)

fig_player_team_winpct.update_traces(
    selector=dict(mode="lines"),
    line=dict(color="crimson", width=3),
    hoverinfo="skip",
    hovertemplate=None
)

fig_player_team_winpct.update_layout(
    template="plotly_white",
    height=650,
    width=1300,
    legend=dict(
        title="Playoff Appearance",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)
#fig_player_team_winpct.show()
fig_player_team_winpct.write_html("team_vorp_vs_win_percentage.html")
display(IFrame("team_vorp_vs_win_percentage.html", width=1300, height=650))


In [40]:
# vorp from drafted players
homegrown_vorp = data[["season", "team", "draft_team", "player_id", "vorp"]].copy()

# Keep only seasons where player was on the team that drafted him
homegrown_vorp = homegrown_vorp[homegrown_vorp["team"] == homegrown_vorp["draft_team"]].copy()
homegrown_vorp_by_team = homegrown_vorp.groupby(["season", "team"], as_index=False).agg(homegrown_vorp=("vorp", "sum"),homegrown_players=("player_id", "nunique"))

# Combine and compute share
team_homegrown_plot = team_vorp_all_players.merge(team_winpct_all_players, on=["season", "team"], how="inner").merge(homegrown_vorp_by_team, on=["season", "team"], how="left")
team_homegrown_plot = team_homegrown_plot.dropna(subset=["homegrown_vorp", "WinPCT"]).copy()
r = team_homegrown_plot["homegrown_vorp"].corr(team_homegrown_plot["WinPCT"])

playoff_plot = team_homegrown_plot.groupby("playoffs", as_index=False).agg(
    avg_homegrown_vorp=("homegrown_vorp", "mean"),
    avg_winpct=("WinPCT", "mean")
)

fig_homegrown_share = go.Figure()

fig_homegrown_share.add_trace(
    go.Scatter(
        x = [1],
        y = [0.5],
        name = "Non-Playoff Teams",
        mode = "markers",
        marker = dict(
            size = 20,
            color = "blue",
            opacity= 0),
        showlegend=True
    )
)
fig_homegrown_share.add_trace(
    go.Scatter(
        x = [1],
        y = [0.5],
        name = "Playoff Teams",
        mode = "markers",
        marker = dict(
            size = 20,
            color = "red",
            opacity= 0),
        showlegend=True
    )
)

fig_homegrown_share.add_trace(
    go.Scatter(
        x=team_homegrown_plot["homegrown_vorp"],
        y=team_homegrown_plot["WinPCT"],
        mode="markers",
        marker=dict(
            size=team_homegrown_plot["homegrown_players"] * 2,
            color=team_homegrown_plot["playoffs"].map({False: "blue", True: "red"}),
            sizemode="area",
            sizeref=2.*max(team_homegrown_plot["homegrown_players"])/(18.**2),
            sizemin=4
        ),
        opacity=0.75,
        customdata = team_homegrown_plot[["season", "team", "team_vorp", "WINS", "LOSSES", "homegrown_players"]].to_numpy(),
        hovertemplate=(
            "<b>%{customdata[1]}</b><br>"
            "Season: %{customdata[0]}<br>"
            "Team Win%: %{y:.3f}<br>"
            "Total Team VORP: %{customdata[2]:.2f}<br>"
            "Own Drafted Players' VORP: %{x:.2f}<br>"
            "Record: %{customdata[3]}-%{customdata[4]}<br>"
            "Homegrown Players: %{customdata[5]:.0f}<br>"
            "<extra></extra>"
        ),
        showlegend=False
        )
        
)

slope, intercept = np.polyfit(team_homegrown_plot['homegrown_vorp'], team_homegrown_plot['WinPCT'], 1)
trendline = slope * team_homegrown_plot['homegrown_vorp'] + intercept

# Add trendline
fig_homegrown_share.add_trace(go.Scatter(x=team_homegrown_plot['homegrown_vorp'], y=trendline, mode='lines', name='Trendline',showlegend=False, line=dict(color='crimson', width=3)))

fig_homegrown_share.add_trace(
    go.Scatter(
        x=playoff_plot["avg_homegrown_vorp"],
        y=playoff_plot["avg_winpct"],
        mode="markers",
        name = "Playoff vs Non-Playoff Averages",
        marker=dict(color="black", sizemode="area", size=20, symbol="diamond", line=dict(width=1.5, color="white")),
        customdata=playoff_plot["playoffs"].map({False: "Non-Playoff Teams", True: "Playoff Teams"}),
        hovertemplate=(
            "<b>%{customdata}</b><br>"
            "Avg Homegrown VORP: %{x:.2f}<br>"
            "Avg Win%: %{y:.3f}<br>"
            "<extra></extra>"
        ),
        showlegend=True
    )
)

fig_homegrown_share.update_traces(
    selector=dict(mode="lines"),
    line=dict(color="crimson", width=3),
    hoverinfo="skip",
    hovertemplate=None
)

fig_homegrown_share.update_layout(
    title=f"Homegrown VORP vs Team Win% (r = {r:.2f})",
    template="plotly_white",
    yaxis_title="Team Win %",
    xaxis_title = "Total Vorp from Homegrown First Round Picks",
    height=650,
    width=1300,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.04,
        xanchor="right",
        x=0.94,
        #font=dict(size=12, color="blue"),
        #bgcolor="LightSteelBlue",
        #bordercolor="Black",
        #borderwidth=1
    )
)
#fig_homegrown_share.show()
fig_homegrown_share.write_html("homegrown_vorp_vs_win_percentage.html")
display(IFrame("homegrown_vorp_vs_win_percentage.html", width=1300, height=650))

In [41]:
box_stats = (
    draft_class_data.groupby("pick_tier")["career_avg_vorp"]
    .agg(
        q1=lambda s: s.quantile(0.25),
        median="median",
        q3=lambda s: s.quantile(0.75),
        min="min",
        max="max",
        count="count",
        iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
        lower_fence=lambda s: s[s >= (s.quantile(0.25) - 1.5 * (s.quantile(0.75) - s.quantile(0.25)))].min(),
        upper_fence=lambda s: s[s <= (s.quantile(0.75) + 1.5 * (s.quantile(0.75) - s.quantile(0.25)))].max(),
    )
    .reset_index()
)
box_stats["pick_tier"] = pd.Categorical(box_stats["pick_tier"],categories=tier_order,ordered=True)
box_stats = box_stats.sort_values("pick_tier")

fig_box_pick_vorp = go.Figure()

# Actual box plot
fig_box_pick_vorp.add_trace(
    go.Box(
        x=draft_class_data["pick_tier"],
        y=draft_class_data["career_avg_vorp"],
        name="Career Avg VORP",
        hoverinfo="skip",
        showlegend=False
    )
)

# Invisible overlay to provide clean custom hover for each box
fig_box_pick_vorp.add_trace(
    go.Bar(
        x=box_stats["pick_tier"],
        y=box_stats["upper_fence"] - box_stats["lower_fence"],
        base=box_stats["lower_fence"],
        width=0.65,
        marker=dict(color="rgba(0,0,0,0.001)", line=dict(width=0)),
        customdata=box_stats[["lower_fence", "q1", "median", "q3", "upper_fence", "count"]].to_numpy(),
        hovertemplate=(
            "Pick: %{x}<br>"
            "Lower Fence: %{customdata[0]:.2f}<br>"
            "Q1: %{customdata[1]:.2f}<br>"
            "Median: %{customdata[2]:.2f}<br>"
            "Q3: %{customdata[3]:.2f}<br>"
            "Upper Fence: %{customdata[4]:.2f}<br>"
            "Players: %{customdata[5]:.0f}<br>"
            "<extra></extra>"
        ),
        showlegend=False
    )
)

fig_box_pick_vorp.update_layout(
    title="Distribution of Career Avg VORP by Overall Pick",
    template="plotly_white",
    height=600,
    width=1300,
    barmode="overlay",
    xaxis_title="Draft Pick",
    yaxis_title=" Career Avg VORP"
)

fig_box_pick_vorp.update_xaxes(categoryorder="array", categoryarray=tier_order)
#fig_box_pick_vorp.write_html("box_plot_career_avg_vorp_by_pick.html")
#display(IFrame("box_plot_career_avg_vorp_by_pick.html", width=1300, height=600))
fig_box_pick_vorp.show()

In [42]:
player_data = draft_class_data.copy()

# all players on playoff teams that played 20+ minutes per game
playoff_players = data[data["playoffs"]==1].copy()
playoff_players['mpg'] = playoff_players["mp"]/playoff_players["g"]
playoff_players = playoff_players[playoff_players['mpg']>=20]
playoff_players = playoff_players.drop_duplicates(subset=["player_id"], keep="first").copy()
playoff_vals = np.sort(playoff_players["vorp"].dropna().to_numpy())

def playoff_rotation_percentile(x, benchmark_sorted):
    if pd.isna(x) or len(benchmark_sorted) == 0:
        return np.nan
    return np.searchsorted(benchmark_sorted, x, side="right") / len(benchmark_sorted)

# base percentile column
player_data["playoff_rotation_percentile"] = player_data["career_avg_vorp"].apply(
    lambda x: playoff_rotation_percentile(x, playoff_vals)
)

# useful derived columns
player_data["rotation_percentile_pct"] = player_data["playoff_rotation_percentile"] * 100
player_data["reaches_rotation_90th"] = player_data["playoff_rotation_percentile"] >= 0.90

pick_summary = (
    player_data.groupby("overall_pick")
    .agg(
        median_playoff_percentile=("playoff_rotation_percentile", "median"),
        mean_playoff_percentile=("playoff_rotation_percentile", "mean"),
        playoff_star_hit_rate=("reaches_rotation_90th", "mean"),
        n_players=("player_id", "count"),
        n_stars=("reaches_rotation_90th", "sum")
    )
    .reset_index()
)

fig_player_pct = go.Figure()

fig_player_pct.add_trace(
    go.Bar(
        x=pick_summary["overall_pick"],
        y=pick_summary["mean_playoff_percentile"],
        name="Mean Playoff Rotation Percentile",
        customdata=pick_summary["n_players"],
        hovertemplate=(
            "Pick: %{x}<br>"
            "Mean Playoff Rotation Percentile: %{y:.1%}<br>"
            "Number of Players: %{customdata}<br>"
            "<extra></extra>"
        )
    )
)
fig_player_pct.add_trace(
    go.Bar(
        x=pick_summary["overall_pick"],
        y=pick_summary["playoff_star_hit_rate"],
        name="Star Hit Rate (90th Percentile+)",
        customdata=pick_summary[ "n_stars"],
        hovertemplate=(
            "Pick: %{x}<br>"
            "Star Hit Rate: %{y:.1%}<br>"
            "Number of Stars: %{customdata}<br>"
            "<extra></extra>"
        )
    )
)

fig_player_pct.update_layout(
    title="How Well Did Draft Picks Perform Relative to Playoff Rotation Players?<br><sup>Draft Pick Career Average VORP vs Playoff Rotation Players Season VORP (Percentiles)</sup><br><sup>Playoff Rotatation Players defined as players with 20+ MPG for a team that made the playoffs</sup>",
    xaxis_title="Overall Pick",
    yaxis_title="Percentile",
    template="plotly_white",
    height=600,
    width=1300,
    barmode='overlay',
    legend=dict(
        yanchor="bottom",
        y=1.04,
        xanchor="right",
        x=0.94,
    )
)


fig_player_pct.update_xaxes(dtick=1)
fig_player_pct.update_yaxes(tickformat=".0%")

fig_player_pct.show()
##fig_player_pct.write_html("draft_pick_playoff_rotation_percentiles.html")
#display(IFrame("draft_pick_playoff_rotation_percentiles.html",width=1300, height=600))


In [43]:
fig_draft_strength = go.Figure()
fig_data = draft_class_data.groupby("season").agg(
    draft_class_career_avg_vorp=("career_avg_vorp", "mean"),
    draft_class_career_total_vorp=("career_total_vorp", "sum"),
    draft_class_rookie_age=("rookie_age", "mean"),
    draft_class_size=("overall_pick", "nunique")
).reset_index()
fig_draft_strength.add_trace(
    go.Bar(
        x=fig_data["season"],
        y=fig_data["draft_class_career_avg_vorp"],
        marker=dict(
            color=fig_data["draft_class_career_total_vorp"],  # Values to use for the color scale
            colorscale='RdYlBu_r',  # Built-in colorscale name
            showscale=True,         # Shows the colorbar on the side
            colorbar=dict(thickness=10, orientation='h', title=dict(text="Total Career VORP of Draft Class", side="top", font=dict(size=12)))
        ),
        customdata=np.stack([
            fig_data["draft_class_rookie_age"],
            fig_data["draft_class_career_total_vorp"],
            fig_data["draft_class_size"]
        ], axis=-1),
        hovertemplate=(
            "Draft Year: %{x}<br>"
            "Draft Class Career Avg VORP: %{y:.2f}<br>"
            "Avg Rookie Age: %{customdata[0]:.1f}<br>"
            "Draft Class Size: %{customdata[2]}<br>"
            "Total Career VORP: %{customdata[1]:.2f}<br>"
            "<extra></extra>"
        )
    )
)

fig_draft_strength.update_layout(
    title="Draft Class Strength",
    xaxis_title="Draft Year",
    yaxis_title="Career Average VORP",
    template="plotly_white",
    height=600,
    width=1300
)
fig_draft_strength.write_html("draft_class_strength.html")
display(IFrame("draft_class_strength.html",width=1300, height=600))

In [44]:
fig_scatter_pick_vorp = px.scatter(
    draft_class_data,
    x="overall_pick",
    y="draft_class_career_avg_vorp_zscore",   
    custom_data=["rookie_player", "season","overall_pick", "first_five_vorp", "seasons_played","draft_class_mean_ff_vorp"],
    labels={
        "overall_pick": "Overall Pick",
        "draft_class_career_avg_vorp_zscore": "Career Avg VORP Z-Score",
        "rookie_player": "Player",
        "season": "Draft Year",
        "seasons_played": "Seasons Played",
        "career_avg_vorp": "Career Avg VORP",
        "draft_class_mean_ff_vorp": "Draft Class Avg First Five-Year VORP"
    },
    title="How well did picks perform relative to their draft class?<br><sup>Draft Pick's Career Avg VORP Z-Score Relative to Draft Class Average</sup>",
    opacity=0.7,
    trendline = "ols"
)

fig_scatter_pick_vorp.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "Player: %{customdata[0]}<br>"
        "Draft Year: %{customdata[1]}<br>"
        "Pick: %{customdata[2]}<br>"
        "Career Avg VORP Z-Score: %{y:.2f}<br>"
        "First Five-Year VORP: %{customdata[3]:.2f}<br>"
        "Seasons Played: %{customdata[4]}<br>"
        "Draft Class Avg First Five-Year VORP: %{customdata[5]:.2f}<br>"
        "<extra></extra>"
    )
)

fig_scatter_pick_vorp.update_traces(
    selector=dict(mode="lines"),
    line=dict(color="red", width=3),
    hoverinfo="skip",
    hovertemplate=None
)


fig_scatter_pick_vorp.add_trace(
    go.Scatter(
        x=draft_class_data["overall_pick"],
        y=draft_class_data["draft_pick_avg_zscore_vs_class"],
        mode="markers",
        marker=dict(size=15, color="red", symbol="x"),
        name="Average Z-Score by Pick",
        hovertemplate="Pick: %{x}<br>Average Z-Score: %{y:.2f}<br><extra></extra>",
        showlegend=False
    )
)

fig_scatter_pick_vorp.update_layout(
    template="plotly_white",
    height=600,
    width=1300,
)

fig_scatter_pick_vorp.update_xaxes(dtick=1,tickvals=list(range(1,31)))

fig_scatter_pick_vorp.show()
#fig_scatter_pick_vorp.write_html("scatter_pick_vs_career_avg_vorp_zscore.html")
#display(IFrame("scatter_pick_vs_career_avg_vorp_zscore.html",width=1300, height=600))

In [ ]:
fig_data = draft_class_data.merge(, on=["season", "team"], how="right")

fig_scatter_pick_vorp = px.scatter(
    fig_data,
    y="winpct_change_after_draft",
    x="player_zscore_diff_from_pick_avg",   
    custom_data=["rookie_player", "season","overall_pick", "first_five_vorp", "seasons_played","draft_class_mean_ff_vorp"],
    labels={
        "overall_pick": "Overall Pick",
        "player_zscore_diff_from_pick_avg": "Career Avg VORP Z-Score vs Pick Avg",
        "rookie_player": "Player",
        "season": "Draft Year",
        "seasons_played": "Seasons Played",
        "career_avg_vorp": "Career Avg VORP",
        "draft_class_mean_ff_vorp": "Draft Class Avg First Five-Year VORP"
    },
    title="How well did picks perform relative to their draft class?<br><sup>Draft Pick's Career Avg VORP Z-Score Relative to Draft Class Average</sup>",
    opacity=0.7,
    trendline = "ols"
)

fig_scatter_pick_vorp.update_traces(
    marker=dict(size=8),
    hovertemplate=(
        "Player: %{customdata[0]}<br>"
        "Draft Year: %{customdata[1]}<br>"
        "Pick: %{customdata[2]}<br>"
        "Career Avg VORP Z-Score: %{y:.2f}<br>"
        "First Five-Year VORP: %{customdata[3]:.2f}<br>"
        "Seasons Played: %{customdata[4]}<br>"
        "Draft Class Avg First Five-Year VORP: %{customdata[5]:.2f}<br>"
        "<extra></extra>"
    )
)

fig_scatter_pick_vorp.update_traces(
    selector=dict(mode="lines"),
    line=dict(color="red", width=3),
    hoverinfo="skip",
    hovertemplate=None
)


fig_scatter_pick_vorp.update_layout(
    template="plotly_white",
    height=600,
    width=1300,
)

fig_scatter_pick_vorp.update_xaxes(dtick=1,tickvals=list(range(1,31)))

fig_scatter_pick_vorp.show()
#fig_scatter_pick_vorp.write_html("scatter_pick_vs_career_avg_vorp_zscore.html")
#display(IFrame("scatter_pick_vs_career_avg_vorp_zscore.html",width=1300, height=600))

ValueError: Value of 'custom_data_0' is not the name of a column in 'data_frame'. Expected one of ['season', 'team', 'WinPCT', 'playoffs', 'playoffs_int', 'winpct_plus_0', 'playoffs_plus_0', 'winpct_plus_1', 'playoffs_plus_1', 'winpct_plus_2', 'playoffs_plus_2', 'winpct_plus_3', 'playoffs_plus_3', 'winpct_plus_4', 'playoffs_plus_4', 'next_5yr_winpct', 'prev_1yr_winpct', 'winpct_change_after_draft', 'next_5yr_playoff_apps', 'n_first_round_picks', 'mean_first_five_vorp', 'best_pick', 'best_pick_tier', 'total_first_five_vorp', 'total_career_vorp', 'best_picks', 'picks_and_players', 'player_zscore_diff_from_pick_avg'] but received: rookie_player

In [ ]:
team_draft_value

,season,team,WinPCT,playoffs,playoffs_int,winpct_plus_0,playoffs_plus_0,winpct_plus_1,playoffs_plus_1,winpct_plus_2,playoffs_plus_2,winpct_plus_3,playoffs_plus_3,winpct_plus_4,playoffs_plus_4,next_5yr_winpct,prev_1yr_winpct,winpct_change_after_draft,next_5yr_playoff_apps,n_first_round_picks,mean_first_five_vorp,best_pick,best_pick_tier,total_first_five_vorp,total_career_vorp,best_picks,picks_and_players
0,1978,Atlanta Hawks,0.561,True,1,0.561,1,0.610,1.0,0.378,0.0,0.512,1.0,0.524,1.0,0.51700,NaN,NaN,4.0,2.0,0.45,10.0,8-14,0.9,0.9,"[10.0, 16.0]","#10 Butch Lee, #16 Jack Givens"
2,1980,Atlanta Hawks,0.378,False,0,0.378,0,0.512,1.0,0.524,1.0,0.488,1.0,0.415,0.0,0.46340,0.610,-0.14660,3.0,1.0,3.60,18.0,15-20,3.6,3.5,[18.0],#18 Don Collins
3,1981,Atlanta Hawks,0.512,True,1,0.512,1,0.524,1.0,0.488,1.0,0.415,0.0,0.610,1.0,0.50980,0.378,0.13180,4.0,1.0,2.00,4.0,4-7,2.0,1.8,[4.0],#4 Al Wood
4,1982,Atlanta Hawks,0.524,True,1,0.524,1,0.488,1.0,0.415,0.0,0.610,1.0,0.695,1.0,0.54640,0.512,0.03440,4.0,2.0,8.80,3.0,1-3,17.6,50.2,"[3.0, 10.0]","#3 Dominique Wilkins, #10 Keith Edmonson"
5,1983,Atlanta Hawks,0.488,True,1,0.488,1,0.415,0.0,0.610,1.0,0.695,1.0,0.610,1.0,0.56360,0.524,0.03960,4.0,1.0,2.60,22.0,21-30,2.6,1.9,[22.0],#22 Randy Wittman
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1296,2015,Washington Wizards,0.500,False,0,0.500,0,0.598,1.0,0.524,1.0,0.390,0.0,0.347,0.0,0.47180,0.561,-0.08920,2.0,1.0,0.30,15.0,15-20,0.3,0.8,[15.0],#15 Kelly Oubre Jr.
1299,2018,Washington Wizards,0.390,False,0,0.390,0,0.347,0.0,0.472,1.0,0.427,0.0,0.427,0.0,0.41260,0.524,-0.11140,1.0,1.0,0.50,15.0,15-20,0.5,0.4,[15.0],#15 Troy Brown Jr.
1300,2019,Washington Wizards,0.347,False,0,0.347,0,0.472,1.0,0.427,0.0,0.427,0.0,0.183,0.0,0.37120,0.390,-0.01880,1.0,1.0,-0.40,9.0,8-14,-0.4,0.1,[9.0],#9 Rui Hachimura
1301,2020,Washington Wizards,0.472,True,1,0.472,1,0.427,0.0,0.427,0.0,0.183,0.0,0.220,0.0,0.34580,0.347,-0.00120,1.0,1.0,2.40,9.0,8-14,2.4,5.6,[9.0],#9 Deni Avdija
